# 🎬 KidoBum Video Generator — LTX-2.3
Free GPU se video generate karo!

**Setup:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run All (Ctrl+F9)
3. Service Account JSON upload karo
4. Video auto-generate hoga!

In [ ]:
# CELL 1: Install
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers accelerate safetensors
!pip install -q imageio[ffmpeg]
!pip install -q google-api-python-client google-auth
print('✅ Done!')

In [ ]:
# CELL 2: Mount Drive
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/KidoBum_Videos'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'✅ Output: {OUTPUT_DIR}')

In [ ]:
# CELL 3: GPU Check
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')
else:
    print('❌ Runtime → Change runtime type → T4 GPU!')

In [ ]:
# CELL 4: Upload Service Account
from google.colab import files
import json

print('📤 Service Account JSON upload karo:')
uploaded = files.upload()
creds_file = list(uploaded.keys())[0]
with open(creds_file) as f:
    creds = json.load(f)
print(f'✅ {creds["client_email"]}')

In [ ]:
# CELL 5: Read Sheet
from google.oauth2 import service_account
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/spreadsheets']
SHEET_ID = '142Y0nesc8iQ2LQ8GFdWBtJCSKK8pCYkKHSlJobhMmhQ'
SHEET_NAME = 'KidoBum_90Day_Prompts.csv'

sa = service_account.Credentials.from_service_account_info(creds, scopes=SCOPES)
sheets = build('sheets', 'v4', credentials=sa).spreadsheets()

result = sheets.values().get(spreadsheetId=SHEET_ID, range=f'{SHEET_NAME}!A:I').execute()
rows = result.get('values', [])

# Find first pending/failed row
pending = None
for idx, row in enumerate(rows[1:], start=2):
    status = row[7].strip().lower() if len(row) > 7 else ''
    title = row[2].strip() if len(row) > 2 else ''
    if not title:
        continue
    if status in ('', 'pending') or 'failed' in status:
        prompts = []
        for r in rows[1:]:
            if len(r) > 6 and r[2].strip() == title and r[6].strip():
                prompts.append(r[6].strip())
        pending = {'row': idx, 'title': title, 'day': row[0], 'prompts': prompts[:3]}
        break

if pending:
    print(f'📋 Day {pending["day"]}: {pending["title"]}')
    print(f'   Prompts: {len(pending["prompts"])}')
else:
    print('❌ No pending rows!')

In [ ]:
# CELL 6: Load LTX-2.3
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

print('📥 Loading LTX-2.3...')
pipe = LTXPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.float16)
pipe.to('cuda')
pipe.enable_model_cpu_offload()
try:
    pipe.enable_xformers_memory_efficient_attention()
except:
    pass
print('✅ Ready!')

In [ ]:
# CELL 7: Generate Video
import time

clips = []
for i, prompt in enumerate(pending['prompts'], 1):
    print(f'\n🎥 Clip {i}: {prompt[:60]}...')
    t0 = time.time()
    
    out = pipe(
        prompt=prompt,
        num_frames=192,        # 8s × 24fps
        num_inference_steps=8,  # Distilled
        guidance_scale=1.0,
        height=768, width=432,  # 9:16 portrait
        generator=torch.Generator('cuda').manual_seed(42+i),
    )
    
    path = f'/content/clip_{i}.mp4'
    export_to_video(out.frames[0], path, fps=24)
    clips.append(path)
    print(f'   ✅ {int(time.time()-t0)}s')
    torch.cuda.empty_cache()

print(f'\n✅ {len(clips)} clips done!')

In [ ]:
# CELL 8: Join + Save
import subprocess

with open('/content/concat.txt', 'w') as f:
    for c in clips:
        f.write(f"file '{c}'\n")

name = pending['title'].replace(' ', '_').replace("'", '')[:50] + '.mp4'
out_path = f'{OUTPUT_DIR}/{name}'

subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', '/content/concat.txt',
    '-c:v', 'libx264', '-preset', 'fast', '-crf', '23',
    '-pix_fmt', 'yuv420p', '-movflags', '+faststart', out_path],
    capture_output=True)

mb = os.path.getsize(out_path) / 1024 / 1024
print(f'✅ Video: {out_path}')
print(f'   Size: {mb:.1f} MB | Duration: {len(clips)*8}s')

In [ ]:
# CELL 9: Update Sheet
sheets.values().update(
    spreadsheetId=SHEET_ID,
    range=f'{SHEET_NAME}!H{pending["row"]}',
    valueInputOption='RAW',
    body={'values': [['done']]}
).execute()

sheets.values().update(
    spreadsheetId=SHEET_ID,
    range=f'{SHEET_NAME}!I{pending["row"]}',
    valueInputOption='RAW',
    body={'values': [[out_path]]}
).execute()

print('✅ Sheet updated: done')
print(f'\n🎬 COMPLETE: {pending["title"]}')
print(f'📁 {out_path}')